# K513 · Week 2, Session 1
## Data visualization — drawing what you learned to read

On last Thursday you learned to *read* a distribution: mean against median, skewness, the IQR, what a
whisker actually is. Today you draw one — and, more usefully, you learn **which picture answers
which question**.

One library on the live path: **seaborn**. Three arguments — `data=`, `x=` / `y=`, and `hue=` — do
almost all of the work, and they mean the same thing in every chart.

Two datasets, both of which you already have: **Boston Housing** and **bikeshare**.

---

### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

There is a specific trap in this session. An AI will happily produce a chart that runs without
error and answers a different question than the one you were asked. Where a cell asks you what a
chart *shows*, that sentence is yours.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one. If
anything ever looks wrong: **Runtime → Restart session and run all**.


---
## 1 · Setup

Three imports and the two data addresses. Run these; neither cell shows anything.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.precision', 3)

In [ ]:
BOSTON_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/BostonHousing.csv"
BIKE_URL   = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/bikeshare.csv"

boston_df = pd.read_csv(BOSTON_URL)
bike_df   = pd.read_csv(BIKE_URL)

`sns` is the conventional short name for seaborn, the way `pd` is for pandas. `plt` is matplotlib —
seaborn is built on top of it, and you will use `plt` occasionally for the size of a figure or to
save one.


---
## 2 · Why draw anything?

You already have `.describe()`. Here is the whole thing for `MEDV`, the column you summarized on
last Thursday.


In [ ]:
boston_df['MEDV'].describe()

Eight numbers, and none of them can tell you the shape. Draw it.


In [ ]:
sns.histplot(data=boston_df, x='MEDV')
plt.show()

Look at the right-hand end. Sixteen tracts sit on **exactly 50.0**, which is not what the tail of a
skewed distribution looks like — it is a wall. The 1970 survey recorded every neighborhood worth
more than \\$50,000 as \\$50,000, so the true values up there are unknown and higher.

That is called **censoring**, and it will quietly break the model you fit in Week 4. `max = 50.00`
in `describe()` gave nothing away. The pile did.


In [ ]:
# how many tracts are sitting exactly on the ceiling?
(boston_df['MEDV'] == 50).sum()

Three defects that summary statistics are structurally blind to, all three in data you have open:

| | What it looks like | Where |
|---|---|---|
| **a ceiling** | values piling up on a boundary | `MEDV` stops at 50 |
| **two humps** | one column, two populations | `RAD` |
| **an impossible value** | one bar off on its own | `humidity` in bikeshare |

Run the next two cells and find the other two for yourself.


In [ ]:
sns.histplot(data=boston_df, x='RAD', bins=24)
plt.show()

In [ ]:
sns.histplot(data=bike_df, x='humidity', bins=40)
plt.show()

`RAD` is the index of accessibility to radial highways. 132 of the 506 tracts sit at 24 and *nothing
at all* sits between 8 and 24 — so this is not one variable with an odd distribution, it is two
kinds of place filed in one table. Its mean of 9.5 describes no tract that exists.

`humidity` has exactly one day recorded at 0%. That does not happen on Earth, and the same row is
labeled `wet`. One broken row, invisible in every number and obvious in the picture.


In [ ]:
bike_df[bike_df['humidity'] == 0]

---
## 3 · One variable, drawn

Two charts. `histplot()` for the shape, `boxplot()` for the extremes.


In [ ]:
sns.histplot(data=boston_df, x='MEDV', bins=30, kde=True)
plt.show()

**`bins=`** is the argument worth arguing about. The default is a guess made by an algorithm that
has never seen your data. The habit is not a number, it is a move: **draw it, change `bins`, draw it
again.** A shape that survives the change is in the data; one that appears at `bins=100` and
vanishes at `bins=20` was never there.

Try it — change the number in the next cell a few times.


In [ ]:
sns.histplot(data=boston_df, x='MEDV', bins=10)
plt.show()

**`kde=True`** adds the smooth curve. It is a smoothing, so it can run past the ends of the real
data — a kde of a variable that cannot be negative will happily show a tail below zero. Useful,
not authoritative.

Now the boxplot. One rule to carry for the rest of the course: **the number you are measuring goes
on `y`.** Later you will add `x=` to split it by a category, and the number stays exactly where it
is.


In [ ]:
sns.boxplot(data=boston_df, y='MEDV')
plt.show()

Last Thursday's five numbers, drawn. The box is Q1 to Q3, the line inside is the median, and the whiskers
stop at **Q3 + 1.5 × IQR** — *not* at the maximum. Everything beyond gets drawn as a dot and called
an outlier.

Those dots at the top are the same sixteen censored tracts from section 2.


In [ ]:
q1 = boston_df['MEDV'].quantile(0.25)
q3 = boston_df['MEDV'].quantile(0.75)
iqr = q3 - q1

print("Q1:              ", q1)
print("Q3:              ", q3)
print("IQR:             ", iqr)
print("whisker ends at: ", q3 + 1.5 * iqr)
print("actual maximum:  ", boston_df['MEDV'].max())

### Why draw both?

Run these two and compare them.


In [ ]:
sns.histplot(data=boston_df, x='RAD', bins=24)
plt.show()

# The one exception to the rule above. This boxplot is drawn on its side so that its
# axis lines up with the histogram directly above it — that alignment is the whole point
# of this comparison. Everywhere else in the course, the number goes on y.
sns.boxplot(data=boston_df, x='RAD')
plt.show()

The histogram shows two clusters and a desert between them. The boxplot shows a wide, unremarkable
box — because a boxplot **is** five numbers, and any distribution with the same five numbers draws
the same box. Bimodality is not one of those five numbers.

That is not a flaw. It is the trade that lets you put ten boxplots side by side, which a histogram
can never do.

> **One variable on its own → histogram.**
> **That variable compared across groups → boxplot.**


---
### ✏️ Now You Try · 1

Use `bike_df`. One row is one day; `num_shared` is how many bikes were rented that day.


**a)** Draw a histogram of `num_shared`. Then change `bins` and draw it again.


In [ ]:
sns.histplot(data=bike_df, x=...)
plt.show()

In [ ]:
# same chart, a different number of bins
...

**b)** Draw a histogram of `humidity`. Something is wrong with this column — what?


In [ ]:
# your code here

**c)** Draw a boxplot of `num_shared`. What is the highest value the whisker reaches?


In [ ]:
# your code here

**d)** `num_shared` has a mean of 4504 and a median of 4548 — almost identical, and its skewness is
−0.05. By every number you learned on last Thursday this column is textbook symmetric.

Draw it. Is it bell-shaped?


In [ ]:
# your code here

**Your answer to (d):**

*(double-click to edit)*


---
## 4 · Comparing groups

Almost every real question is a comparison. Three charts and one argument.


In [ ]:
sns.countplot(data=bike_df, x='weather')
plt.show()

`countplot()` counts rows for you — no `value_counts()` first, no `groupby`. It is the count half of
an Excel pivot table, drawn.

Use `x=` for vertical bars and `y=` for horizontal. If the category names are long, use `y=`; never
rotate the labels.

Now the confusion this session exists to prevent.


In [ ]:
sns.barplot(data=bike_df, x='weather', y='num_shared', errorbar=None)
plt.show()

Same x-axis, completely different question.

| | Question | Bar height |
|---|---|---|
| `countplot(x='weather')` | how many wet **days** were there? | a count of rows — 21 |
| `barplot(x='weather', y='num_shared')` | how many **bikes** go out on a wet day? | the mean of `num_shared` — 1,803 |

**The tell: `countplot()` takes one variable. `barplot()` takes two.** If the call has both an `x`
and a `y`, the bar height is a statistic, not a count.

`errorbar=None` turns off the 95% confidence interval seaborn draws by default. `estimator='median'`
switches the statistic — worth it for a skewed column.


In [ ]:
sns.barplot(data=bike_df, x='weather', y='num_shared', errorbar=None, estimator='median')
plt.show()

### The workhorse chart

A boxplot with a category added on `x=` — the number stays on `y`. Of everything in this session, this is the one you will use
most.


In [ ]:
sns.boxplot(data=bike_df, x='weather', y='num_shared')
plt.show()

Three things are on that chart, and only the first is on a bar chart of the same data:

1. Wet days are dramatically worse — median 1,817 against 4,844.
2. **The boxes overlap.** The top whisker on `wet` reaches about 4,600, which is above the *median*
   clear day. "Wet days are worse" is true of the middle and false of a good quarter of wet days.
3. The wet box is short — less variable, not just lower.

A barplot would have given you three numbers and invited the sentence *"wet days get 1,800 rides"*,
which is wrong about a lot of wet days.

### `hue` — one argument, a second comparison


In [ ]:
sns.boxplot(data=bike_df, x='weather', y='num_shared', hue='busyday')
plt.show()

`hue` works identically on `histplot`, `boxplot`, `countplot`, `barplot`, `scatterplot` and
`pairplot` — same argument, same meaning, legend built for you.

Now look for what is **not** there. There is no orange box over `wet`: in 731 days, not one busy day
was a wet day. A table of group medians would simply not have had that row, and a missing row in a
table is almost impossible to see. A missing box in a row of boxes is obvious.

> A chart can show you something that is not there. A table can only show you what is.


In [ ]:
# the same fact, as a table — notice you have to go looking for the NaN
bike_df.pivot_table(index='weather', columns='busyday', values='num_shared', aggfunc='median')

---
### ✏️ Now You Try · 2

Still `bike_df`. `weekday` runs 0 (Sunday) to 6 (Saturday).


**a)** Draw a countplot of `weekday`. What do you expect to see *before* you run it?


In [ ]:
# your code here

**b)** Draw a barplot of the mean `num_shared` by `weekday`. Turn the error bars off.


In [ ]:
sns.barplot(data=bike_df, x=..., y=..., errorbar=...)
plt.show()

**c)** Draw a boxplot of `num_shared` split by `weekday`. Does the day of the week matter?


In [ ]:
# your code here

**d)** Add `hue='weather'` to (c). Of the two — day of the week, or weather — which would you plan
staffing around?


In [ ]:
# your code here

**Your answer to (d):**

*(double-click to edit)*


---
## 5 · Two variables at once


In [ ]:
sns.scatterplot(data=bike_df, x='temp in Celsius', y='num_shared')
plt.show()

One dot per row — 731 days, 731 dots. Convention: put the thing you think *explains* on `x`, and the
thing being *explained* on `y`. Keep the habit and Week 4 will feel familiar.

With hundreds of points, solid markers hide how many are stacked underneath. `alpha=` fixes that in
one argument.


In [ ]:
sns.scatterplot(data=bike_df, x='temp in Celsius', y='num_shared', alpha=0.5)
plt.show()

The correlation between `temp in Celsius` and `num_shared` is **0.63** — "warmer days, more riders."
Look hard at the right-hand end of the chart before you accept that sentence.


In [ ]:
bike_df['temp in Celsius'].corr(bike_df['num_shared'])

In [ ]:
# mean rides in each 5-degree band
bins = [0, 10, 15, 20, 25, 30, 40]
bike_df.groupby(pd.cut(bike_df['temp in Celsius'], bins), observed=True)['num_shared'].mean().round(0)

Ridership climbs to about 25 °C and then **falls**. Too hot is as bad as too cold.

*r* measures exactly one thing: how close the cloud of dots is to a **straight line**. A
relationship that rises and then falls is real, strong and useful — and *r* will quietly understate
it. A model that only knew *r* would recommend that the ideal bike-share day is the hottest one.

> Compute *r*. Then draw the picture anyway. We read *r* properly in Week 3.


### Screening tools

`pairplot()` draws every scatterplot at once. **Always pass `vars=`** — without it seaborn draws
every numeric column against every other, and thirty columns is 900 panels.


In [ ]:
sns.pairplot(data=bike_df,
             vars=['temp in Celsius', 'humidity', 'windspeed', 'num_shared'],
             hue='weather')
plt.show()

The diagonal is each variable's own distribution, so a pairplot is section 3 and section 5 in one
picture. It is a screening tool — it tells you which panel is worth drawing properly. It is not the
chart that goes in the report.

`heatmap()` does the same job for the correlations. `corr()` works on numeric columns only, so
select the columns you want first.


In [ ]:
cols = ['temp in Celsius', 'humidity', 'windspeed', 'num_shared']
sns.heatmap(data=bike_df[cols].corr(), annot=True)
plt.show()

`annot=True` prints the values on the squares. Without them a heatmap is a poster.

Note that `temp in Celsius` and `num_shared` sit at 0.63 on this matrix — and you have just seen with
your own eyes that 0.63 is hiding a curve. A heatmap is a shortlist of pairs worth drawing, not a
set of conclusions.


---
### ✏️ Now You Try · 3

Back to `boston_df`. `LSTAT` is the percentage of the population classed as lower status; `RM` is
the average number of rooms per dwelling.


**a)** Draw a scatterplot of `MEDV` against `LSTAT`. Is it a straight line?


In [ ]:
sns.scatterplot(data=boston_df, x=..., y=...)
plt.show()

**b)** Add `hue='CHAS'`. Does being on the river explain the dots sitting above the trend?


In [ ]:
# your code here

**c)** Draw a pairplot of `MEDV`, `RM`, `LSTAT` and `CRIM`. Which pair looks most linear?


In [ ]:
# your code here

**d)** Draw the heatmap of those four columns. Does the strongest number match the panel you picked
in (c)?


In [ ]:
# your code here

**Your answer to (d):**

*(double-click to edit)*


---
## Choosing a chart

The whole session on one table. Start from the **question**, not from the chart.

| Your question | What you have | The chart |
|---|---|---|
| What does this one column look like? | one continuous | `histplot()` |
| How extreme are the extremes? | one continuous | `boxplot(y=)` |
| How many of each kind? | one category | `countplot()` |
| Is this number different across groups? | continuous + category | `boxplot(x=, y=)` |
| What is the average, per group? | continuous + category | `barplot(x=, y=)` |
| Do these two move together? | two continuous | `scatterplot()` |
| Which pairs are worth a proper look? | many continuous | `pairplot()` |
| How strong is every pair, at a glance? | many continuous | `heatmap()` |

Add **`hue=`** to any of them to split by one more category.


---
## Making a chart presentable

Every seaborn call returns a matplotlib *axes* object, which is why all of this works.


In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.boxplot(data=bike_df, x='weather', y='num_shared', order=['clear', 'cloudy', 'wet'])
ax.set_title('Daily ridership by weather, 2011-2012')
ax.set_xlabel('weather')
ax.set_ylabel('rides per day')
plt.show()

- `plt.figure(figsize=(w, h))` **before** the seaborn call. Width first, in inches.
- `order=[...]` fixes the order the categories appear in. Without it seaborn uses whatever order it
  met them in the data.
- `ax.set_title(...)`, `ax.set_xlabel(...)`, `ax.set_ylabel(...)`. Seaborn labels the axis with the
  column name, which is rarely what a reader needs. `num_shared` means *rides per day*; say so.
- `plt.savefig('chart.png', dpi=200, bbox_inches='tight')` before `plt.show()`, if you need the file.


---
## That's it

You can now open a dataset you have never seen and show somebody what is in it. Specifically, you:

- found a ceiling, a bimodal column and a broken row — three defects no summary statistic can show
- drew a histogram and a boxplot, and can say which one to reach for and why
- told `countplot()` from `barplot()` by counting the variables in the call
- used `hue=` to turn any chart into a comparison, and read a *missing* box as a finding
- saw a relationship that *r* understates, and know to draw the picture anyway

### Before Thursday

The **Week 2 assignment** is on Canvas, due **Sunday 11:59 pm**. Finish any Now You Try cells you
did not reach — the assignment assumes you did them.

Thursday is **data transformation**: the tables you called untidy in Week 1 are the ones you fix.
